# Single-Country Endowment Economy Baseline

This notebook simulates the **single-country endowment OLG economy** that underlies
the two-country bubble model in Hirano et al. (2025).

### Model
Young saves $A_t = \beta e_t$ in the Lucas tree (market clearing: $Q^u_t = \beta e^u_t$).
Regime switching: u-state ($g_{e,u}$, bubble persists) vs absorbing b-state ($g_{e,b}$, bubble pops).

**Key extension**: b-state price decays as $Q^b_t = \beta\,\lambda_e\,\rho^t\,e^u_t$,
where $\rho \in (0,1]$ is a per-period decay factor.

| $\rho$ | Interpretation | $C^b/C^u$ asymptote | Condition 2b |
|---|---|---|---|
| 1 | constant $\lambda_e$ (old spec) | $\lambda_e$ (constant) | **diverges** |
| $g_{e,b}/g_{e,u}$ | b-price on fundamental growth path | $\to 0$ geometrically | **converges** ✓ |

**Natural default**: $\rho = g_{e,b}/g_{e,u}$ so that $Q^b_t = \beta e_0 g_{e,b}^t$
(regardless of switch date, the b-state price equals what the endowment *would* have been
under the lower fundamental growth rate).

In [ ]:
include("SingleCountryOLG.jl")
using Plots, Printf
default(fontfamily="Computer Modern", linewidth=2, framestyle=:box, legend=:topright)

## 1. Comparing ρ = 1 (no bubble) vs ρ = g_e_b/g_e_u (bubble exists)

In [ ]:
# ρ = 1: old constant-λ_e spec  (condition 2b diverges)
p1_no  = SCParams(ρ = 1.0)
r_no   = run_sc_simulation(p1_no; verbose=true)

println()

# ρ = g_e_b/g_e_u: natural default  (condition 2b converges)
p_def  = SCParams()          # ρ = 1.01^5/1.025^5 ≈ 0.929 by default
r_def  = run_sc_simulation(p_def; verbose=true)

## 2. Exogenous paths (same for both cases)

In [ ]:
T   = p_def.T_max
ts  = 1:T
ts2 = 2:T

e_path = [r_def.states[t].e for t in ts]
D_path = [r_def.states[t].D for t in ts]
d_path = [r_def.states[t].d for t in ts]

pa = plot(ts, e_path, label="e^u_t", yscale=:log10,
          title="Exogenous paths (log scale)",
          xlabel="Period t", ylabel="Level")
plot!(pa, ts, D_path, label="D^u_t", linestyle=:dash)

pb = plot(ts, d_path, label="d_t = D_t/e_t  (→ 0)",
          title="Dividend-endowment ratio d_t",
          xlabel="Period t", ylabel="D/e")

plot(pa, pb, layout=(1,2), size=(900,350))

$d_t$ declines at rate $g_{D,u}/g_{e,u} \approx 0.952$ per period → **Condition 2a** converges.

## 3. Effect of ρ on the b-state price and consumption ratio

In [ ]:
# b-state scaling λ_e_t = λ_e · ρ^t over time
λet_no  = [r_no.states[t].λ_e_t  for t in ts]
λet_def = [r_def.states[t].λ_e_t for t in ts]

# Consumption ratio C^b/C^u = R_b/R_u
cr_no  = [r_no.diagnostics.consump_ratio[t]  for t in ts2]
cr_def = [r_def.diagnostics.consump_ratio[t] for t in ts2]

pc = plot(ts, λet_no,  label="ρ=1  (constant)",
          title="Effective b-state scaling λ_e · ρ^t",
          xlabel="Period t", ylabel="λ_e · ρ^t")
plot!(pc, ts, λet_def, label=@sprintf("ρ = g_e_b/g_e_u ≈ %.3f", p_def.ρ),
      linestyle=:dash)
hline!(pc, [0.0], color=:black, linestyle=:dot, label="")

pd = plot(ts2, cr_no,  label="ρ=1  →  C^b/C^u → λ_e=1",
          title="Consumption ratio C^b_t / C^u_t",
          xlabel="Period t", ylabel="Ratio")
plot!(pd, ts2, cr_def,
      label=@sprintf("ρ≈%.3f  →  C^b/C^u → 0", p_def.ρ), linestyle=:dash)
hline!(pd, [1.0], color=:grey, linestyle=:dot, label="1")
hline!(pd, [0.0], color=:black, linestyle=:dot, label="0")

plot(pc, pd, layout=(1,2), size=(900,350))

**Key**: With $\rho = g_{e,b}/g_{e,u}$, the effective scaling $\lambda_e\rho^t \to 0$,
driving $C^b/C^u \to 0$.  The ratio $(C^b/C^u)^{1-\gamma}$ then decays geometrically
at rate $\rho^{1-\gamma}$, making the sum in Condition 2b finite.

## 4. Bubble diagnostic conditions

In [ ]:
# Condition 2a (same for both — depends only on dividends)
sum2a = r_def.diagnostics.sum_dividend_ratio

# Condition 2b terms and cumulative sums
terms_no  = r_no.diagnostics.cond_2b_terms
terms_def = r_def.diagnostics.cond_2b_terms
sum2b_no  = r_no.diagnostics.sum_cond_2b
sum2b_def = r_def.diagnostics.sum_cond_2b

pe = plot(ts, sum2a[ts],
          label="Σ d_t  (condition 2a)",
          title="Condition 2a converges ✓",
          xlabel="Period t", ylabel="Cumulative Σ d_t")

pf = plot(ts2, sum2b_no[ts2],
          label="ρ=1  (diverges)",
          title="Condition 2b",
          xlabel="Period t", ylabel="Cumulative Σ (C^b/C^u)^{1-γ}")
plot!(pf, ts2, sum2b_def[ts2],
      label=@sprintf("ρ≈%.3f  (converges ✓)", p_def.ρ), linestyle=:dash)

# Overlay individual terms to show geometric decay
pg = plot(ts2, terms_no[ts2],
          label="ρ=1",
          title="Individual terms (C^b/C^u)^{1-γ}",
          xlabel="Period t", ylabel="Term value")
plot!(pg, ts2, terms_def[ts2],
      label=@sprintf("ρ≈%.3f", p_def.ρ), linestyle=:dash)
# Expected geometric decay reference
ργ = p_def.ρ^(1 - p_def.γ)
plot!(pg, ts2, ργ .^ ts2, color=:grey, linestyle=:dot,
      label=@sprintf("(ρ^{1-γ})^t = %.3f^t", ργ))

plot(pe, pf, pg, layout=(1,3), size=(1200,350))

**Left**: Condition 2a converges in both cases (dividends shrink relative to endowment).

**Middle**: Condition 2b — with $\rho=1$ the sum grows linearly; with $\rho = g_{e,b}/g_{e,u}$ it converges.

**Right**: Individual terms $(C^b/C^u)^{1-\gamma}$ decay as $(\rho^{1-\gamma})^t \approx 0.964^t$
— a convergent geometric series.

## 5. ρ sweep: transition from diverging to converging

In [ ]:
ρ_vals  = [1.0, 0.97, 0.95, p_def.ρ, 0.85]
colors  = [:steelblue, :darkorange, :forestgreen, :crimson, :purple]

ph = plot(title="Condition 2b cumulative sum for various ρ",
          xlabel="Period t", ylabel="Σ (C^b/C^u)^{1-γ}", legend=:topleft)
pi_ = plot(title="C^b/C^u ratio for various ρ",
           xlabel="Period t", ylabel="C^b_t / C^u_t", legend=:topright)

for (ρv, col) in zip(ρ_vals, colors)
    ri   = run_sc_simulation(SCParams(ρ=ρv); verbose=false)
    conv = ri.diagnostics.condition_2b_converges
    lbl  = @sprintf("ρ=%.3f  (%s)", ρv, conv ? "converges" : "diverges")
    plot!(ph, ts2, ri.diagnostics.sum_cond_2b[ts2], label=lbl, color=col)
    plot!(pi_, ts2, ri.diagnostics.consump_ratio[ts2], label=lbl, color=col)
end
hline!(pi_, [0.0], color=:black, linestyle=:dot, label="")

plot(ph, pi_, layout=(1,2), size=(1000,380))

**Interpretation**: Any $\rho < 1$ causes $C^b/C^u \to 0$, making condition 2b converge
(provided $\gamma < 1$ so that $1-\gamma > 0$ and the geometric ratio $\rho^{1-\gamma} < 1$).

The natural choice $\rho = g_{e,b}/g_{e,u}$ has a clean interpretation:
> *When the bubble pops at date $t$, the stock price reverts to $Q^b_t = \beta e_0 g_{e,b}^t$,
> the endowment level that would have prevailed had the economy always grown at the
> slower balanced-growth rate $g_{e,b}$.*

The gap $(g_{e,u}/g_{e,b})^t \to \infty$ represents the accumulated "bubble premium"
in the u-path endowment — the larger the premium, the more catastrophic the pop.

In [ ]:
# Summary table
println("Summary: bubble existence by ρ")
println("─" ^ 55)
@printf("%-8s  %-10s  %-10s  %-12s  %s\n",
        "ρ", "2a conv", "2b conv", "bubble", "C^b/C^u at T")
println("─" ^ 55)
for ρv in ρ_vals
    ri = run_sc_simulation(SCParams(ρ=ρv); verbose=false)
    d  = ri.diagnostics
    @printf("%-8.4f  %-10s  %-10s  %-12s  %.6f\n",
            ρv, d.condition_2a_converges, d.condition_2b_converges,
            d.bubble_exists, d.consump_ratio[end])
end

## 6. Deterministic bubble burst at τ = 50

We now simulate a **deterministic** path where the economy is in the u-state for
$t = 1, \ldots, 49$ and the bubble bursts at $t = 50$.

**Stock price along this path**:
$$
Q_t = \begin{cases}
Q^u_t = \beta e^u_t & t < 50 \\
Q^b_t = \beta \lambda_e \rho^t e^u_t = \beta e_0 g_{e,b}^t & t \geq 50
\end{cases}
$$

**Fundamental value** (= b-state price regardless of burst date):
$$F_t = Q^b_t = \beta e_0 g_{e,b}^t$$

**Bubble component**:
$$B_t = Q_t - F_t =
\begin{cases}
\beta e_0\!\left(g_{e,u}^t - g_{e,b}^t\right) > 0 & t < 50 \\
0 & t \geq 50
\end{cases}$$
The bubble collapses discontinuously at $\tau = 50$, dropping by $B_{49} = \beta e_0(g_{e,u}^{49} - g_{e,b}^{49})$.

In [ ]:
τ   = 50          # bubble bursts at t = τ
p   = SCParams()  # natural ρ = g_e_b/g_e_u
paths  = generate_sc_paths(p)
states = solve_sc_u_path(p, paths)

T  = p.T_max
ts = 1:T

# Stock price: u-path for t < τ, b-path at t ≥ τ
Q_path = [t < τ ? states[t].Q_u : states[t].Q_b for t in ts]

# Fundamental value F_t = Q^b_t = β·e_0·g_e_b^t  (independent of τ)
F_path = [states[t].Q_b for t in ts]

# Bubble component B_t = Q_t - F_t  (= 0 for t ≥ τ)
B_path = Q_path .- F_path

# ── Print key values ──────────────────────────────────────────────────
println("Deterministic burst at τ = $τ")
println("─"^55)
@printf("%-6s  %-12s  %-12s  %-12s\n", "t", "Q_t", "F_t", "B_t")
println("─"^55)
for t in [1, 10, 25, 40, 48, 49, 50, 51, 60, 80, 100, 120]
    @printf("%-6d  %-12.4f  %-12.4f  %-12.4f\n",
            t, Q_path[t], F_path[t], B_path[t])
end
println()
@printf("B_{%d} (just before burst) = %.4f  →  B_{%d} (just after) = %.4f\n",
        τ-1, B_path[τ-1], τ, B_path[τ])
@printf("Instantaneous price drop  = %.4f  (%.1f%% of Q_{%d})\n",
        B_path[τ-1], 100*B_path[τ-1]/Q_path[τ-1], τ-1)

# ── Plots ─────────────────────────────────────────────────────────────
# 1. Stock price decomposition (log scale)
p1 = plot(ts, Q_path, label="Q_t (actual)",
          title="Stock Price (log scale)",
          xlabel="Period t", ylabel="Q_t",
          yscale=:log10, color=:steelblue, linewidth=2)
plot!(p1, ts, F_path, label="F_t (fundamental)",
      linestyle=:dash, color=:forestgreen, linewidth=2)
vline!(p1, [τ], color=:red, linestyle=:dot, label="burst τ=$τ", linewidth=1.5)

# 2. Bubble component level
p2 = plot(ts, B_path, label="B_t",
          title="Bubble Component  B_t = Q_t − F_t",
          xlabel="Period t", ylabel="B_t",
          color=:crimson, fill=0, fillalpha=0.25, linewidth=2)
vline!(p2, [τ], color=:red, linestyle=:dot, label="burst τ=$τ", linewidth=1.5)

# 3. Bubble share B_t / Q_t (only pre-burst)
bubble_share = B_path ./ max.(Q_path, 1e-15)
p3 = plot(1:(τ-1), bubble_share[1:(τ-1)],
          label="B_t / Q_t",
          title="Bubble Share of Stock Price",
          xlabel="Period t", ylabel="B_t / Q_t",
          color=:darkorange, ylims=(0, 1), linewidth=2)
vline!(p3, [τ], color=:red, linestyle=:dot, label="burst τ=$τ", linewidth=1.5)

# 4. Stacked area: fundamental vs bubble, full horizon
p4 = plot(ts, F_path, fillrange=0, fillalpha=0.5,
          label="Fundamental F_t", color=:forestgreen,
          title="Price Decomposition (stacked)",
          xlabel="Period t", ylabel="Q_t", linewidth=0)
plot!(p4, ts, Q_path, fillrange=F_path, fillalpha=0.5,
      label="Bubble B_t", color=:crimson, linewidth=0)
plot!(p4, ts, Q_path, label="Q_t", color=:black, linewidth=1.5, linestyle=:solid)
vline!(p4, [τ], color=:red, linestyle=:dot, label="burst τ=$τ", linewidth=1.5)

plot(p1, p2, p3, p4, layout=(2, 2), size=(1100, 700))